# Hierarchical Data Loader

In [1]:
from os import chdir
from pathlib import Path

cwd = Path.cwd()
print(f"CWD: {cwd}")
if cwd.name == "code":
    chdir("..")
print(f"CWD: {Path.cwd()}")

CWD: /home/james/git/research/fed-learning/code
CWD: /home/james/git/research/fed-learning


In [2]:
from torch.utils.data import Dataset, DataLoader, TensorDataset, Sampler, IterableDataset, ChainDataset
import torch
from torch.optim import SGD

from tqdm.notebook import tqdm
import pandas as pd
import numpy as np
from time import perf_counter

In [3]:
from src.training.decentralized import decentralized_train_loop, decentralized_validate_loop
from src.models.MatrixFactorization import MF, UMF
from src.graphs import random_k_out_graph
from src.users import User
from src.data_utils import create_dataloader

In [4]:
def chunks(lst, n):
    """Yield successive n-sized chunks from lst."""
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

def double_chunks(X, y, n):
    for i in range(0, X.shape[0], n):
        yield X[i:i+n], y[i:i+n]

class MyIterableDataset(IterableDataset):

    def __init__(self, X, y, batch_size=3, shuffle=True):
        super().__init__()
        self.X = X
        self.y = y
        self.batch_size = batch_size
        self.shuffle = shuffle

    def __iter__(self):
        n = self.X.shape[0]
        if self.shuffle:
            idx = torch.randperm(n)
        else:
            idx = range(n)
        return double_chunks(self.X[idx], self.y[idx], self.batch_size)#, chunks(self.y, self.batch_size)

class ChainDatasetShuffle(IterableDataset):
    
    def __init__(self, datasets, shuffle=True):
        super().__init__()
        self.datasets = datasets
        self.shuffle = shuffle

    def __iter__(self):
        if self.shuffle:
            idx = torch.randperm(len(self.datasets))
            for i in idx:
                yield from self.datasets[i]
            # for d in self.datasets[idx]:
            #     yield from d
        else:
            for d in self.datasets:
                yield from d
        

    def __len__(self):
        total = 0
        for d in self.datasets:
            assert isinstance(d, IterableDataset), "ChainDataset only supports IterableDataset"
            total += len(d)  # type: ignore[arg-type]
        return total
                
def create_batched_dataloaders(train_df, batch_size=5):
    datasets = []
    for g, _df in train_df.groupby("user_id"):
        x = _df[["user_id", "item_id"]].to_numpy()
        y = _df["rating"].to_numpy()
        datasets.append(MyIterableDataset(torch.tensor(x), torch.tensor(y), batch_size=batch_size))
    cdataset = ChainDatasetShuffle(datasets)
    return DataLoader(cdataset)


# class UserRandomSamplingDataset(Dataset):
#     def __init__(self, x: np.ndarray, y: np.ndarray):
#         self.x = torch.tensor(x)
#         self.y = torch.tensor(y)
#         self.y = self.y.unsqueeze(-1) if self.y.ndim == 1 else self.y

#     def __len__(self):
#         return len(self.x[:, 0].unique())

#     def __getitem__(self, user):
#         user_idx = self.x[:, 0] == user
#         x = self.x[user_idx, :]
#         y = self.y[user_idx]
#         selected_indices = torch.randint(x.shape[0], size=(10,))
#         return x[selected_indices, :], y[selected_indices]
        

In [5]:
train_df = pd.read_csv("dataset/ml100k_train.csv")
test_df = pd.read_csv("dataset/ml100k_test.csv")
X_train = train_df[["user_id", "item_id"]]
y_train = train_df["rating"]
X_test = test_df[["user_id", "item_id"]]
y_test = test_df["rating"]
n_users = train_df.user_id.nunique()
n_items = train_df.item_id.nunique()

In [6]:
graph = random_k_out_graph(n_users,5,50)

In [11]:
oaat_train_dl = create_dataloader(train_df, "oaat")
urs1_train_dl = create_dataloader(train_df, "rs", batch_size=1)
urs2_train_dl = create_dataloader(train_df, "rs", batch_size=2)
urs10_train_dl = create_dataloader(train_df, "rs", batch_size=10)
urs_test_dl = create_dataloader(test_df, "oaat")

## Compare different dl.s

In [8]:
n_epochs = 5

In [22]:
start = perf_counter()
users = {}
for user in range(n_users):
    model = MF(n_users, n_items, 30)
    users[user] = User(user, model, SGD(model.parameters(), lr=.01, weight_decay=.001))

for i in tqdm(range(n_epochs)):
    train_loss = decentralized_train_loop(users, oaat_train_dl, graph)
    val_loss = decentralized_validate_loop(users, urs_test_dl)
    print(val_loss)
print(f"Time: {perf_counter() - start}")

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/75000 [00:00<?, ?it/s]

2.8072463500221883


  0%|          | 0/75000 [00:00<?, ?it/s]

2.578137600913743


...

2.5314795855075403


  0%|          | 0/75000 [00:00<?, ?it/s]

2.514939210547812


  0%|          | 0/75000 [00:00<?, ?it/s]

2.5062932963239426
Time: 933.6736149229982


In [12]:
start = perf_counter()

users = {}
for user in range(n_users):
    model = MF(n_users, n_items, 30)
    users[user] = User(user, model, SGD(model.parameters(), lr=.01, weight_decay=.001))


for i in tqdm(range(n_epochs)):
    train_loss = decentralized_train_loop(users, urs1_train_dl, graph)
    val_loss = decentralized_validate_loop(users, urs_test_dl)
    print(val_loss)
print(f"Time: {perf_counter() - start}")

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/75000 [00:00<?, ?it/s]

3.22335294633067


  0%|          | 0/75000 [00:00<?, ?it/s]

2.778018995642578


...

2.6337609416314285


  0%|          | 0/75000 [00:00<?, ?it/s]

2.5714137617948807


  0%|          | 0/75000 [00:00<?, ?it/s]

2.5418231883040407
Time: 925.1748850990116


In [14]:
start = perf_counter()

users = {}
for user in range(n_users):
    model = MF(n_users, n_items, 30)
    users[user] = User(user, model, SGD(model.parameters(), lr=.01/2, weight_decay=.001))


for i in tqdm(range(n_epochs)):
    train_loss = decentralized_train_loop(users, urs2_train_dl, graph)
    val_loss = decentralized_validate_loop(users, urs_test_dl)
    print(val_loss)
print(f"Time: {perf_counter() - start}")

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/75000 [00:00<?, ?it/s]

3.7605427698906357


  0%|          | 0/75000 [00:00<?, ?it/s]

3.2208851394058695


...

3.016330665225271


  0%|          | 0/75000 [00:00<?, ?it/s]

2.918116717325981


  0%|          | 0/75000 [00:00<?, ?it/s]

2.8656768126832883
Time: 950.0078377210011


In [16]:
start = perf_counter()

users = {}
for user in range(n_users):
    model = MF(n_users, n_items, 30)
    users[user] = User(user, model, SGD(model.parameters(), lr=.01 /2, weight_decay=.001))

for i in tqdm(range(n_epochs)):
    train_loss = decentralized_train_loop(users, urs10_train_dl, graph)
    val_loss = decentralized_validate_loop(users, urs_test_dl)
    print(val_loss)
print(f"Time: {perf_counter() - start}")

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/75000 [00:00<?, ?it/s]

3.696111312349562


  0%|          | 0/75000 [00:00<?, ?it/s]

3.1838103896972627


...

2.992031988927185


  0%|          | 0/75000 [00:00<?, ?it/s]

2.900483716386942


  0%|          | 0/75000 [00:00<?, ?it/s]

2.849594097524267
Time: 1127.6623413320049


In [11]:
decentralized_validate_loop(users, urs_test_dl)

2.779979358705329

In [11]:
for idx, (x, y) in enumerate(urs_train_dl):
    print(x, x.shape)
    print(y, y.shape)
    if idx > 3:
        break

IndexError: too many indices for tensor of dimension 1

In [ ]:
bdl = create_batched_dataloaders(train_df, batch_size=5)

In [ ]:
for idx, (x, y) in enumerate(bdl):
    print(x, x.shape)
    print(y, y.shape)
    if idx > 3:
        break

In [ ]:
datasets = []
for g, _df in train_df.groupby("user_id"):
    x = _df[["user_id", "item_id"]].to_numpy()
    y = _df["rating"].to_numpy()
    datasets.append(MyIterableDataset(torch.tensor(x), torch.tensor(y), batch_size=2))

In [ ]:
cdataset = ChainDatasetShuffle(datasets)
cdloader = DataLoader(cdataset)

In [ ]:
prev_user = -1
for idx, (x, y) in enumerate(cdloader):
    if idx < 3:
        print(x, x.shape)
        print(y, y.shape)
    if x[0, 0,0] != prev_user:
        prev_user = x[0,0,0]
        print(f"new user: {x[0, 0,0]}")

In [ ]:
bs = 10
idataset = MyIterableDataset(X_train.to_numpy(), y_train.to_numpy(), batch_size=bs, shuffle=True)
idataloader = DataLoader(idataset, batch_size=3, shuffle=False)

In [ ]:
import math
for idx, (x, y) in enumerate(idataloader):
    if idx < 2:
        print(x, math.ceil(x.shape[0] / 3))
        print(y, y.shape)
    

In [ ]:
for idx, (x, y) in enumerate(idl):
    if idx < 5:
        print(x)
        print(y)
        print(type(x))
        print(type(y))

In [ ]:
class HierarchicalDataLoader:
    def __init__(self, datasets, batch_size=1, shuffle=True):
        self.datasets = datasets,
        self.batch_size = batch_size
        self.shuffle = shuffle

    

In [ ]:
## Try on 